In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [2]:
def generate_teacher_data(teacher, num_samples=1000):
    X = torch.randn(num_samples, teacher.fc1.in_features)
    with torch.no_grad():
        y = teacher(X)
    return X, y

In [3]:
class Teacher(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.init_weights()
        
    def init_weights(self):
        for param in self.parameters():
            param.data.uniform_(-1, 1)
    
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

In [4]:
input_size = 100
hidden_teacher = 50
output_size = 10

In [5]:
teacher = Teacher(input_size, hidden_teacher, output_size)
X, y = generate_teacher_data(teacher)

In [ ]:
X.shape, y.shape

In [7]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(X, y)

train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [8]:
class MLP(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16, output_size=10):
        super(MLP, self).__init__()
        self.fc0 = nn.Linear(input_size, hidden_size)
        self.fc1 = nn.Linear(hidden_size, output_size)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.fc1(self.act(self.fc0(x)))

In [9]:
model = MLP(input_size=input_size, hidden_size=hidden_teacher, output_size=output_size)
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0, model.fc1], device=device)

In [10]:
hyperparams = {
    "num_epochs": 512,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.55},
    "pruning_thresholds": {"fc0": 0.2},
    "plateau_threshold": 1,
    "min_epochs_between_expansions": 64,
    "plateau_window_size": 5,
    "learning_rate": 0.01,
    "prune_after_epochs": 2,
    "task_type": "regression",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 900,
    "weight_decay": 0
}

In [ ]:
import wandb
wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-teacher-student",
    name=f"name",
    config=hyperparams
)

In [ ]:
import time
start_time = time.time()

criterion = nn.MSELoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, val_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))